In [4]:
import torch
import matplotlib.pyplot as plt

from metrics import (
    align_channels,
    abs_error_map,
    mse_map,
    darcy_residual_map,
)


CHANNEL_NAMES = {
    0: "K",
    1: "P",
    2: "phi",
}


def unpack_dataset_item(item):
    """
    Handles both:
        Full dataset: feat, label
        Limited dataset: feat, label, mask
    """
    if len(item) == 3:
        feat, label, mask = item
    else:
        feat, label = item
        mask = None

    return feat, label, mask


def get_model_prediction(model, feat, device):
    model.eval()

    with torch.no_grad():
        feat_b = feat.unsqueeze(0).to(device)
        pred = model(feat_b).cpu().squeeze(0)

    return pred


def compute_heatmap_for_item(
    model,
    dataset,
    idx,
    device,
    metric="abs_error",
    channel=0,
):
    """
    Produces one heatmap for one dataset item.

    metric options:
        "abs_error"
        "squared_error"
        "darcy_pred"
        "darcy_true"
        "darcy_match"
    """

    item = dataset[idx]
    feat, label, mask = unpack_dataset_item(item)

    pred = get_model_prediction(model, feat, device)

    # Add batch dimension for metric functions
    pred_b = pred.unsqueeze(0)
    label_b = label.unsqueeze(0)

    label_b = align_channels(label_b, pred_b)

    if metric == "abs_error":
        heat = abs_error_map(pred_b, label_b)[0, channel]

    elif metric == "squared_error":
        heat = mse_map(pred_b, label_b)[0, channel]

    elif metric == "darcy_pred":
        heat = torch.abs(darcy_residual_map(pred_b))[0, 0]

    elif metric == "darcy_true":
        heat = torch.abs(darcy_residual_map(label_b))[0, 0]

    elif metric == "darcy_match":
        heat = darcy_match_map(pred_b, label_b)[0, 0]

    else:
        raise ValueError(f"Unknown metric: {metric}")

    return heat


def average_heatmap(
    model,
    dataset,
    indices,
    device,
    metric="abs_error",
    channel=0,
):
    """
    Averages heatmaps across multiple dataset indices.
    """

    heatmaps = []

    for idx in indices:
        heat = compute_heatmap_for_item(
            model=model,
            dataset=dataset,
            idx=idx,
            device=device,
            metric=metric,
            channel=channel,
        )
        heatmaps.append(heat)

    return torch.stack(heatmaps).mean(dim=0)


def plot_heatmap(
    heatmap,
    title="Heatmap",
    save_path=None,
):
    plt.figure(figsize=(6, 5))
    plt.imshow(heatmap.numpy())
    plt.colorbar()
    plt.title(title)
    plt.axis("off")

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", dpi=200)

    plt.show()


def plot_single_model_heatmap(
    model,
    dataset,
    indices,
    device,
    metric="abs_error",
    channel=0,
    save_path=None,
):
    """
    Main function you probably want to call.

    Example:
        plot_single_model_heatmap(
            model,
            val_data,
            indices=range(0, 50),
            device=DEVICE,
            metric="abs_error",
            channel=0
        )
    """

    heatmap = average_heatmap(
        model=model,
        dataset=dataset,
        indices=indices,
        device=device,
        metric=metric,
        channel=channel,
    )

    if metric in ["abs_error", "squared_error"]:
        channel_name = CHANNEL_NAMES.get(channel, f"channel {channel}")
        title = f"Average {metric} heatmap for {channel_name}"
    else:
        title = f"Average {metric} heatmap"

    plot_heatmap(
        heatmap,
        title=title,
        save_path=save_path,
    )

    return heatmap

In [5]:
import torch
import datasets
import matplotlib.pyplot as plt

from model_loader import make_model, DEVICE

model = make_model(
    model_type="attn_unet",
    channels="KP"
)
model.load_state_dict(torch.load(
    "official_darcy/final/border_physics_limited_attn_unet_darcy_5p0_final_state.pt",
    # "baseline_full/final/fixed_attn_unet_baseline_full_nodarcy_best_state.pt",
    # "recent_analysis_unet/physics_tests/fixed/fixed_physics_limited_attn_unet_darcy_0p0_best_state.pt",
    map_location=DEVICE
))

model.eval()

import numpy as np
import torch
import matplotlib.pyplot as plt
import datasets
from model_loader import make_model, DEVICE, ChannelSelectDataset

val_sims = np.array([250])

base_dataset = datasets.BorderDenseDatasetLimited(
    sims=val_sims,
    channels="KP"
)

dataset = ChannelSelectDataset(base_dataset, channels="KP")

C:\Users\jaden\AppData\Local\Temp\ipykernel_24356\1563974057.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(


In [6]:
heatmap = plot_single_model_heatmap(
    model=model,
    dataset=base_dataset,
    indices=range(0, 50),
    device=DEVICE,
    metric="abs_error",
    channel=0,   # 0 = K, 1 = P
    save_path="heatmap_abs_error_K.png"
)

RuntimeError: Given groups=1, weight of size [8, 2, 3, 3], expected input[1, 3, 200, 200] to have 2 channels, but got 3 channels instead